# Phase 0 go/no-go on Colab (Qwen3.5-4B)

Thin launcher: all logic lives in the repository. Runtime: **GPU (L4 or A100)**.
Run all cells; the last one prints a summary to paste back. Safe to re-run from the top.

In [ ]:
REPO_URL = "https://github.com/desire-del/faithful-question-summarization.git"
REF = "main"  # use a tag (e.g. v0.0) once it exists
%cd /content
!rm -rf repo && git clone -q --depth 1 --branch $REF $REPO_URL repo
%cd /content/repo
# Colab preinstalls torchao 0.10, which peft rejects (it needs >= 0.16). This project does not use it.
!pip uninstall -y -q torchao
!pip install -q -e '.[train]'

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1) Generation: text-only loading, thinking disabled, latency
!python scripts/go_no_go/smoke_generate.py --env colab --model qwen35_4b

In [ ]:
# 2) GRPO: a few TRL + LoRA steps, peak memory and time per step
!python scripts/go_no_go/grpo_smoke.py --env colab --model qwen35_4b --steps 4

In [ ]:
# 3) Summary to paste back
import json, pathlib
for path in sorted(pathlib.Path('reports/go_no_go').glob('*_colab.json')):
    report = json.loads(path.read_text())
    keep = ['decision', 'error', 'model_class', 'peak_memory_gb', 'steady_step_seconds',
            'estimated_full_run_hours', 'generate_seconds_batch', 'thinking_leaked']
    summary = {k: report[k] for k in keep if k in report}
    summary['gpu'] = (report.get('provenance', {}).get('gpu') or {}).get('name')
    summary['packages'] = report.get('provenance', {}).get('packages')
    print(path.name, json.dumps(summary, indent=1))
    for sample in report.get('samples', [])[:3]:
        print('  ->', sample['output'])